# Unseen Test Set Evaluation

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class MulticlassTargetEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, smoothing=10):
        self.smoothing = smoothing

    def fit(self, X, y):
        X = pd.DataFrame(X).copy()
        y = pd.Series(y).reset_index(drop=True)
        X = X.reset_index(drop=True)

        data = pd.DataFrame({
            "service": X["service"],
            "target": y
        })

        self.classes_ = sorted(y.unique())

        self.global_mean_ = {
            cls: (y == cls).mean()
            for cls in self.classes_
        }

        self.mapping_ = {}

        for cls in self.classes_:

            stats = data.groupby("service")["target"].agg(
                count="count",
                mean=lambda x, c=cls: (x == c).mean()
            )

            smoothing_weight = (
                stats["count"] /
                (stats["count"] + self.smoothing)
            )

            encoded = (
                smoothing_weight * stats["mean"]
                + (1 - smoothing_weight) * self.global_mean_[cls]
            )

            self.mapping_[cls] = encoded.to_dict()

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        result = pd.DataFrame(index=X.index)

        for cls in self.classes_:
            result[f"service_te_{cls}"] = (
                X["service"]
                .map(self.mapping_[cls])
                .fillna(self.global_mean_[cls])
                .values
            )

        return result

In [20]:
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [2]:
TEST_PATH = "../data/raw/KDDTest+.txt"
PREPROCESSOR_PATH = "../data/processed/preprocessor.joblib"

print("Paths configured successfully.")

Paths configured successfully.


In [3]:
preprocessor = joblib.load(PREPROCESSOR_PATH)

print("Preprocessor loaded successfully.")

Preprocessor loaded successfully.


In [4]:
test_df = pd.read_csv(
    TEST_PATH,
    sep="\t",
    header=None
)

print("Test shape:", test_df.shape)
test_df.head()

Test shape: (22544, 42)


,0,1,2,3,4,5,6,7,8,9,...,32,33,34,35,36,37,38,39,40,41
0,0,tcp,private,REJ,0,0,0,0,0,0,...,10,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,Attack
1,0,tcp,private,REJ,0,0,0,0,0,0,...,1,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,Attack
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,86,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,Normal
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,57,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,Attack
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,86,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,Attack


In [5]:
feature_names_original = [
    "duration", "protocol_type", "service", "flag",
    "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent",
    "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count",
    "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate"
]

test_df.columns = feature_names_original + ["label"]

print("Columns assigned:", len(test_df.columns))

Columns assigned: 42


In [6]:
X_test = test_df[feature_names_original].copy()
y_test = test_df["label"].copy()

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (22544, 41)
y_test shape: (22544,)


In [23]:
y_test_binary = y_test.map({
    "Normal": 0,
    "Attack": 1
})

print(y_test_binary.value_counts())

label
1    12833
0     9711
Name: count, dtype: int64


In [24]:
print("Missing values:", X_test.isnull().sum().sum())
print("Unique labels:", y_test_binary.unique())
print("Feature count:", X_test.shape[1])

Missing values: 0
Unique labels: [1 0]
Feature count: 41


In [25]:
X_test_processed = preprocessor.transform(X_test)

print("Processed test shape:", X_test_processed.shape)

Processed test shape: (22544, 121)


In [26]:
print("Test features:", X_test_processed.shape[1])
print("Expected features:", 121)

Test features: 121
Expected features: 121


In [27]:
print("Any missing values:", np.isnan(X_test_processed).sum())
print("Any infinite values:", np.isinf(X_test_processed).sum())

Any missing values: 0
Any infinite values: 0


In [28]:
MODEL_PATH = "../models/random_forest_binary.joblib"

random_forest_binary = joblib.load(MODEL_PATH)

print("Binary Random Forest loaded successfully.")

Binary Random Forest loaded successfully.


In [29]:
print("Test samples:", X_test_processed.shape[0])
print("Test features:", X_test_processed.shape[1])
print("Model:", random_forest_binary)

Test samples: 22544
Test features: 121
Model: RandomForestClassifier(class_weight='balanced', n_estimators=200, n_jobs=-1,
                       random_state=42)


In [30]:
y_test_pred = random_forest_binary.predict(X_test_processed)

print("Predictions generated:", len(y_test_pred))

Predictions generated: 22544


In [34]:
test_accuracy = accuracy_score(y_test_binary, y_test_pred)
test_precision = precision_score(y_test_binary, y_test_pred)
test_recall = recall_score(y_test_binary, y_test_pred)
test_f1 = f1_score(y_test_binary, y_test_pred)

print("Test Accuracy :", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall   :", test_recall)
print("Test F1-score :", test_f1)

Test Accuracy : 0.7702714691270405
Test Precision: 0.968765311121999
Test Recall   : 0.6163017221226526
Test F1-score : 0.7533457160546745


In [35]:
print(
    classification_report(
        y_test_binary,
        y_test_pred,
        target_names=["Normal", "Attack"]
    )
)

              precision    recall  f1-score   support

      Normal       0.66      0.97      0.79      9711
      Attack       0.97      0.62      0.75     12833

    accuracy                           0.77     22544
   macro avg       0.81      0.80      0.77     22544
weighted avg       0.83      0.77      0.77     22544



In [37]:
cm_test = confusion_matrix(
    y_test_binary,
    y_test_pred,
    labels=[0,1]
)

print(cm_test)

[[9456  255]
 [4924 7909]]


In [38]:
print("Training/validation label distribution:")
print(y_test_binary.value_counts(normalize=True).sort_index())

Training/validation label distribution:
label
0    0.430758
1    0.569242
Name: proportion, dtype: float64


In [39]:
print("Individual labels in KDDTest+:")
print(y_test.value_counts())

Individual labels in KDDTest+:
label
Attack    12833
Normal     9711
Name: count, dtype: int64


In [40]:
results_df = pd.DataFrame({
    "Actual": y_test_binary,
    "Predicted": y_test_pred
})

print(pd.crosstab(
    results_df["Actual"],
    results_df["Predicted"],
    normalize="index"
))

Predicted         0         1
Actual                       
0          0.973741  0.026259
1          0.383698  0.616302


In [41]:
false_negatives = results_df[
    (results_df["Actual"] == 1) &
    (results_df["Predicted"] == 0)
]

print("False negatives:", len(false_negatives))

False negatives: 4924


In [42]:
train_df = pd.read_csv(
    "../data/raw/KDDTrain+.txt",
    sep="\t",
    header=None
)

train_df.columns = feature_names_original + ["label", "difficulty"]

y_train_binary = train_df["label"].apply(
    lambda x: 0 if x == "normal" else 1
)

print("Training samples:", len(train_df))
print("Training class distribution:")
print(y_train_binary.value_counts())
print(y_train_binary.value_counts(normalize=True))

Training samples: 125973
Training class distribution:
label
0    67343
1    58630
Name: count, dtype: int64
label
0    0.534583
1    0.465417
Name: proportion, dtype: float64


In [43]:
print("TRAIN")
print(y_train_binary.value_counts(normalize=True))

print("\nTEST")
print(y_test_binary.value_counts(normalize=True))

TRAIN
label
0    0.534583
1    0.465417
Name: proportion, dtype: float64

TEST
label
1    0.569242
0    0.430758
Name: proportion, dtype: float64


In [44]:
from scipy.stats import ks_2samp

numerical_features_test = [
    feature for feature in feature_names_original
    if train_df[feature].dtype != "object"
]

ks_results = []

for feature in numerical_features_test:
    statistic, p_value = ks_2samp(
        train_df[feature],
        test_df[feature]
    )

    ks_results.append({
        "Feature": feature,
        "KS Statistic": statistic,
        "p-value": p_value
    })

ks_df = pd.DataFrame(ks_results).sort_values(
    "KS Statistic",
    ascending=False
)

ks_df.head(20)

,Feature,KS Statistic,p-value
39,dst_host_rerror_rate,0.226030,0.000000e+00
37,dst_host_serror_rate,0.202509,0.000000e+00
38,dst_host_srv_serror_rate,0.196824,0.000000e+00
24,serror_rate,0.185392,0.000000e+00
25,srv_serror_rate,0.181381,0.000000e+00
40,dst_host_srv_rerror_rate,0.167978,0.000000e+00
26,rerror_rate,0.130280,1.946068e-283
5,dst_bytes,0.124082,4.627402e-257
27,srv_rerror_rate,0.123649,2.869508e-255
3,flag,0.113037,2.782944e-213


In [45]:
significant_features = ks_df[
    ks_df["p-value"] < 0.05
]

print(
    "Features with statistically significant distribution difference:",
    len(significant_features),
    "out of",
    len(ks_df)
)

print("\nLargest distribution differences:")
print(significant_features.head(15))

Features with statistically significant distribution difference: 29 out of 41

Largest distribution differences:
                     Feature  KS Statistic        p-value
39      dst_host_rerror_rate      0.226030   0.000000e+00
37      dst_host_serror_rate      0.202509   0.000000e+00
38  dst_host_srv_serror_rate      0.196824   0.000000e+00
24               serror_rate      0.185392   0.000000e+00
25           srv_serror_rate      0.181381   0.000000e+00
40  dst_host_srv_rerror_rate      0.167978   0.000000e+00
26               rerror_rate      0.130280  1.946068e-283
5                  dst_bytes      0.124082  4.627402e-257
27           srv_rerror_rate      0.123649  2.869508e-255
3                       flag      0.113037  2.782944e-213
32        dst_host_srv_count      0.110666  1.999011e-204
33    dst_host_same_srv_rate      0.104691  6.147313e-183
34    dst_host_diff_srv_rate      0.103573  4.707372e-179
29             diff_srv_rate      0.097910  5.152251e-160
2                

In [47]:
categorical_features = [
    "protocol_type",
    "service",
    "flag"
]

numerical_features_test = [
    feature
    for feature in feature_names_original
    if feature not in categorical_features
]

print("Numerical features:", len(numerical_features_test))
print("Categorical features:", len(categorical_features))

Numerical features: 38
Categorical features: 3


In [48]:
from scipy.stats import ks_2samp

ks_results = []

for feature in numerical_features_test:
    statistic, p_value = ks_2samp(
        train_df[feature],
        test_df[feature]
    )

    ks_results.append({
        "Feature": feature,
        "KS Statistic": statistic,
        "p-value": p_value
    })

ks_df = pd.DataFrame(ks_results).sort_values(
    "KS Statistic",
    ascending=False
).reset_index(drop=True)

ks_df.head(20)

,Feature,KS Statistic,p-value
0,dst_host_rerror_rate,0.226030,0.000000e+00
1,dst_host_serror_rate,0.202509,0.000000e+00
2,dst_host_srv_serror_rate,0.196824,0.000000e+00
3,serror_rate,0.185392,0.000000e+00
4,srv_serror_rate,0.181381,0.000000e+00
5,dst_host_srv_rerror_rate,0.167978,0.000000e+00
6,rerror_rate,0.130280,1.946068e-283
7,dst_bytes,0.124082,4.627402e-257
8,srv_rerror_rate,0.123649,2.869508e-255
9,dst_host_srv_count,0.110666,1.999011e-204


In [49]:
top_shift_features = ks_df.head(10)["Feature"].tolist()

distribution_comparison = []

for feature in top_shift_features:
    distribution_comparison.append({
        "Feature": feature,
        "Train Mean": train_df[feature].mean(),
        "Test Mean": test_df[feature].mean(),
        "Train Median": train_df[feature].median(),
        "Test Median": test_df[feature].median(),
        "KS Statistic": ks_df.loc[
            ks_df["Feature"] == feature,
            "KS Statistic"
        ].iloc[0]
    })

distribution_comparison_df = pd.DataFrame(
    distribution_comparison
)

distribution_comparison_df

,Feature,Train Mean,Test Mean,Train Median,Test Median,KS Statistic
0,dst_host_rerror_rate,0.118832,0.233385,0.0,0.0,0.226030
1,dst_host_serror_rate,0.284452,0.097814,0.0,0.0,0.202509
2,dst_host_srv_serror_rate,0.278485,0.099426,0.0,0.0,0.196824
3,serror_rate,0.284485,0.102924,0.0,0.0,0.185392
4,srv_serror_rate,0.282485,0.103635,0.0,0.0,0.181381
5,dst_host_srv_rerror_rate,0.120240,0.226683,0.0,0.0,0.167978
6,rerror_rate,0.119958,0.238463,0.0,0.0,0.130280
7,dst_bytes,19779.114421,2056.018808,0.0,46.0,0.124082
8,srv_rerror_rate,0.121183,0.235179,0.0,0.0,0.123649
9,dst_host_srv_count,115.653005,140.750532,63.0,168.0,0.110666


In [50]:
significant_features = ks_df[
    ks_df["p-value"] < 0.05
]

print(
    "Numerical features with significant distribution difference:",
    len(significant_features),
    "out of",
    len(ks_df)
)

print("\nLargest distribution differences:")
print(significant_features.head(15))

Numerical features with significant distribution difference: 26 out of 38

Largest distribution differences:
                     Feature  KS Statistic        p-value
0       dst_host_rerror_rate      0.226030   0.000000e+00
1       dst_host_serror_rate      0.202509   0.000000e+00
2   dst_host_srv_serror_rate      0.196824   0.000000e+00
3                serror_rate      0.185392   0.000000e+00
4            srv_serror_rate      0.181381   0.000000e+00
5   dst_host_srv_rerror_rate      0.167978   0.000000e+00
6                rerror_rate      0.130280  1.946068e-283
7                  dst_bytes      0.124082  4.627402e-257
8            srv_rerror_rate      0.123649  2.869508e-255
9         dst_host_srv_count      0.110666  1.999011e-204
10    dst_host_same_srv_rate      0.104691  6.147313e-183
11    dst_host_diff_srv_rate      0.103573  4.707372e-179
12             diff_srv_rate      0.097910  5.152251e-160
13             same_srv_rate      0.094669  1.327931e-149
14                   

In [51]:
# Separate Normal and Attack samples
train_normal = train_df[train_df["label"] == "normal"]
train_attack = train_df[train_df["label"] != "normal"]

test_normal = test_df[test_df["label"] == "Normal"]
test_attack = test_df[test_df["label"] == "Attack"]

print("Train Normal :", len(train_normal))
print("Test Normal  :", len(test_normal))
print("Train Attack :", len(train_attack))
print("Test Attack  :", len(test_attack))


# Compare Normal distributions
normal_shift = []

for feature in numerical_features_test:
    statistic, p_value = ks_2samp(
        train_normal[feature],
        test_normal[feature]
    )

    normal_shift.append({
        "Feature": feature,
        "KS Statistic": statistic,
        "p-value": p_value
    })

normal_shift_df = pd.DataFrame(normal_shift).sort_values(
    "KS Statistic",
    ascending=False
)


# Compare Attack distributions
attack_shift = []

for feature in numerical_features_test:
    statistic, p_value = ks_2samp(
        train_attack[feature],
        test_attack[feature]
    )

    attack_shift.append({
        "Feature": feature,
        "KS Statistic": statistic,
        "p-value": p_value
    })

attack_shift_df = pd.DataFrame(attack_shift).sort_values(
    "KS Statistic",
    ascending=False
)


# Summary
print("\nNormal — Top 15 distribution shifts:")
print(normal_shift_df.head(15))

print("\nAttack — Top 15 distribution shifts:")
print(attack_shift_df.head(15))

print(
    "\nNormal features with significant shift:",
    (normal_shift_df["p-value"] < 0.05).sum(),
    "out of 38"
)

print(
    "Attack features with significant shift:",
    (attack_shift_df["p-value"] < 0.05).sum(),
    "out of 38"
)

Train Normal : 67343
Test Normal  : 9711
Train Attack : 58630
Test Attack  : 12833

Normal — Top 15 distribution shifts:
                     Feature  KS Statistic        p-value
30    dst_host_same_srv_rate      0.159388  8.182853e-189
31    dst_host_diff_srv_rate      0.158563  7.395508e-187
29        dst_host_srv_count      0.151529  1.329187e-170
20                 srv_count      0.137880  3.345823e-141
19                     count      0.109555   3.599866e-89
2                  dst_bytes      0.106775   1.030709e-84
1                  src_bytes      0.090935   1.603564e-61
0                   duration      0.075112   4.343445e-42
37  dst_host_srv_rerror_rate      0.060768   1.095937e-27
28            dst_host_count      0.055313   5.206578e-23
35  dst_host_srv_serror_rate      0.050462   3.197881e-19
27        srv_diff_host_rate      0.048933   4.237852e-18
8                  logged_in      0.046228   3.364334e-16
24           srv_rerror_rate      0.043039   4.246612e-14
34      d

In [52]:
for feature in categorical_features:

    train_dist = train_df[feature].value_counts(normalize=True)
    test_dist = test_df[feature].value_counts(normalize=True)

    comparison = pd.concat(
        [train_dist.rename("Train"),
         test_dist.rename("Test")],
        axis=1
    ).fillna(0)

    comparison["Absolute Difference"] = (
        comparison["Train"] - comparison["Test"]
    ).abs()

    print(f"\n===== {feature} =====")
    print(
        comparison.sort_values(
            "Absolute Difference",
            ascending=False
        ).head(15)
    )


===== protocol_type =====
                  Train      Test  Absolute Difference
protocol_type                                         
tcp            0.815167  0.837473             0.022307
icmp           0.065816  0.046265             0.019551
udp            0.119018  0.116262             0.002756

===== service =====
             Train      Test  Absolute Difference
service                                          
telnet    0.018679  0.072126             0.053447
pop_3     0.002096  0.045200             0.043105
private   0.173474  0.211764             0.038290
domain_u  0.071785  0.039656             0.032129
http      0.320211  0.348341             0.028130
eco_i     0.036405  0.011622             0.024783
ftp       0.013924  0.030696             0.016772
ftp_data  0.054456  0.037748             0.016708
smtp      0.058052  0.041430             0.016622
Attack4   0.000000  0.013573             0.013573
ecr_i     0.024426  0.033357             0.008931
finger    0.014027  0.00603

In [53]:
false_negative_data = test_df.loc[
    (y_test_binary == 1) & (y_test_pred == 0)
].copy()

print("Total false negatives:", len(false_negative_data))

print("\nTop services among false negatives:")
print(
    false_negative_data["service"]
    .value_counts(normalize=True)
    .head(15)
)

print("\nTop flags among false negatives:")
print(
    false_negative_data["flag"]
    .value_counts(normalize=True)
    .head(10)
)

Total false negatives: 4924

Top services among false negatives:
service
telnet      0.282291
pop_3       0.182372
ftp         0.121040
private     0.101137
http        0.099919
ftp_data    0.068034
smtp        0.060723
Attack4     0.034119
other       0.029448
sunrpc      0.011170
pm_dump     0.002843
X11         0.002031
finger      0.001219
domain      0.001219
auth        0.000812
Name: proportion, dtype: float64

Top flags among false negatives:
flag
SF      0.712835
RSTO    0.074736
REJ     0.060723
S0      0.050772
S3      0.049553
RSTR    0.048741
S1      0.001828
S2      0.000812
Name: proportion, dtype: float64


In [54]:
# Load the tuned binary Random Forest

tuned_rf_binary = joblib.load(
    "../models/random_forest_binary_tuned.joblib"
)

# Predict on the unseen KDDTest+ set

y_test_pred_tuned = tuned_rf_binary.predict(X_test_processed)

# Evaluate

tuned_test_accuracy = accuracy_score(
    y_test_binary,
    y_test_pred_tuned
)

tuned_test_precision = precision_score(
    y_test_binary,
    y_test_pred_tuned
)

tuned_test_recall = recall_score(
    y_test_binary,
    y_test_pred_tuned
)

tuned_test_f1 = f1_score(
    y_test_binary,
    y_test_pred_tuned
)

print("Tuned Random Forest - KDDTest+")
print("--------------------------------")
print("Accuracy :", tuned_test_accuracy)
print("Precision:", tuned_test_precision)
print("Recall   :", tuned_test_recall)
print("F1       :", tuned_test_f1)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_binary, y_test_pred_tuned))

Tuned Random Forest - KDDTest+
--------------------------------
Accuracy : 0.7687189496096523
Precision: 0.9685155577419752
Recall   : 0.6136523026572118
F1       : 0.751287922152261

Confusion Matrix:
[[9455  256]
 [4958 7875]]


In [60]:
# Apply the frozen validation threshold to KDDTest+

TEST_THRESHOLD = 0.35

y_test_proba_tuned = tuned_rf_binary.predict_proba(
    X_test_processed
)[:, 1]

y_test_pred_threshold = (
    y_test_proba_tuned >= TEST_THRESHOLD
).astype(int)

print("Threshold:", TEST_THRESHOLD)

print("\nKDDTest+ Results")
print("----------------")

print(
    "Accuracy :",
    accuracy_score(y_test_binary, y_test_pred_threshold)
)

print(
    "Precision:",
    precision_score(
        y_test_binary,
        y_test_pred_threshold
    )
)

print(
    "Recall   :",
    recall_score(
        y_test_binary,
        y_test_pred_threshold
    )
)

print(
    "F1       :",
    f1_score(
        y_test_binary,
        y_test_pred_threshold
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_binary,
        y_test_pred_threshold
    )
)

Threshold: 0.35

KDDTest+ Results
----------------
Accuracy : 0.8138307310149042
Precision: 0.9688382193268187
Recall   : 0.695316761474324
F1       : 0.8095994193167899

Confusion Matrix:
[[9424  287]
 [3910 8923]]


In [61]:
print("Unique test labels:")
print(sorted(y_test.unique()))

print("\nLabel counts:")
print(y_test.value_counts())

Unique test labels:
['Attack', 'Normal']

Label counts:
label
Attack    12833
Normal     9711
Name: count, dtype: int64


In [65]:
# Load the full attack-type-labelled KDDTest+ file

TEST_PATH_MULTICLASS = "../data/raw/KDDTest+2.txt"

test_multi_df = pd.read_csv(
    TEST_PATH_MULTICLASS,
    sep=",",
    header=None
)

print("Test shape:", test_multi_df.shape)

Test shape: (22544, 43)


In [66]:
# Assign columns

test_multi_df.columns = (
    feature_names_original
    + ["label", "difficulty"]
)

print("Unique labels:")
print(sorted(test_multi_df["label"].unique()))

print("\nLabel counts:")
print(test_multi_df["label"].value_counts())

Unique labels:
['apache2', 'back', 'buffer_overflow', 'ftp_write', 'guess_passwd', 'httptunnel', 'imap', 'ipsweep', 'land', 'loadmodule', 'mailbomb', 'mscan', 'multihop', 'named', 'neptune', 'nmap', 'normal', 'perl', 'phf', 'pod', 'portsweep', 'processtable', 'ps', 'rootkit', 'saint', 'satan', 'sendmail', 'smurf', 'snmpgetattack', 'snmpguess', 'sqlattack', 'teardrop', 'udpstorm', 'warezmaster', 'worm', 'xlock', 'xsnoop', 'xterm']

Label counts:
label
normal             9711
neptune            4657
guess_passwd       1231
mscan               996
warezmaster         944
apache2             737
satan               735
processtable        685
smurf               665
back                359
snmpguess           331
saint               319
mailbomb            293
snmpgetattack       178
portsweep           157
ipsweep             141
httptunnel          133
nmap                 73
pod                  41
buffer_overflow      20
multihop             18
named                17
ps               

In [67]:
attack_category_map_test = {
    # DoS
    "back": "DoS",
    "land": "DoS",
    "neptune": "DoS",
    "pod": "DoS",
    "smurf": "DoS",
    "teardrop": "DoS",
    "apache2": "DoS",
    "mailbomb": "DoS",
    "processtable": "DoS",
    "udpstorm": "DoS",

    # Probe
    "ipsweep": "Probe",
    "nmap": "Probe",
    "portsweep": "Probe",
    "satan": "Probe",
    "mscan": "Probe",
    "saint": "Probe",

    # R2L
    "ftp_write": "R2L",
    "guess_passwd": "R2L",
    "imap": "R2L",
    "multihop": "R2L",
    "phf": "R2L",
    "spy": "R2L",
    "warezclient": "R2L",
    "warezmaster": "R2L",
    "named": "R2L",
    "sendmail": "R2L",
    "snmpgetattack": "R2L",
    "snmpguess": "R2L",
    "worm": "R2L",
    "xlock": "R2L",
    "xsnoop": "R2L",
    "httptunnel": "R2L",

    # U2R
    "buffer_overflow": "U2R",
    "loadmodule": "U2R",
    "perl": "U2R",
    "rootkit": "U2R",
    "ps": "U2R",
    "sqlattack": "U2R",
    "xterm": "U2R",

    # Normal
    "normal": "Normal"
}

test_multi_df["attack_category"] = (
    test_multi_df["label"].map(attack_category_map_test)
)

print("Unmapped labels:")
print(
    test_multi_df.loc[
        test_multi_df["attack_category"].isna(),
        "label"
    ].unique()
)

Unmapped labels:
<StringArray>
[]
Length: 0, dtype: str


In [68]:
X_test_multi = test_multi_df[
    feature_names_original
].copy()

y_test_multi_binary = (
    test_multi_df["attack_category"] != "Normal"
).astype(int)

y_test_multi_class = (
    test_multi_df["attack_category"]
)

X_test_multi_processed = preprocessor.transform(
    X_test_multi
)

print("Processed test shape:", X_test_multi_processed.shape)
print("Binary target shape:", y_test_multi_binary.shape)
print("Multiclass target shape:", y_test_multi_class.shape)

Processed test shape: (22544, 121)
Binary target shape: (22544,)
Multiclass target shape: (22544,)


In [69]:
y_test_binary_proba_full = (
    tuned_rf_binary.predict_proba(
        X_test_multi_processed
    )[:, 1]
)

TEST_THRESHOLD = 0.35

y_test_binary_pred_full = (
    y_test_binary_proba_full >= TEST_THRESHOLD
).astype(int)

print("Binary KDDTest+2 Results")
print("------------------------")
print("Accuracy :", accuracy_score(
    y_test_multi_binary, y_test_binary_pred_full
))
print("Precision:", precision_score(
    y_test_multi_binary, y_test_binary_pred_full
))
print("Recall   :", recall_score(
    y_test_multi_binary, y_test_binary_pred_full
))
print("F1       :", f1_score(
    y_test_multi_binary, y_test_binary_pred_full
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_multi_binary, y_test_binary_pred_full
))

Binary KDDTest+2 Results
------------------------
Accuracy : 0.813875088715401
Precision: 0.9688416024318749
Recall   : 0.6953946855762487
F1       : 0.8096534204318635

Confusion Matrix:
[[9424  287]
 [3909 8924]]


In [71]:
# Load the final tuned multiclass Random Forest

final_random_forest = joblib.load(
    "../models/random_forest_multiclass_final.joblib"
)

print("Multiclass Random Forest loaded.")

Multiclass Random Forest loaded.


In [72]:
y_test_multiclass_pred = (
    final_random_forest.predict(
        X_test_multi_processed
    )
)

print("Multiclass KDDTest+2 Results")
print("----------------------------")

print("Accuracy:",
      accuracy_score(
          y_test_multi_class,
          y_test_multiclass_pred
      ))

print("\nClassification Report:")
print(
    classification_report(
        y_test_multi_class,
        y_test_multiclass_pred,
        labels=["Normal", "DoS", "Probe", "R2L", "U2R"],
        zero_division=0
    )
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test_multi_class,
        y_test_multiclass_pred,
        labels=["Normal", "DoS", "Probe", "R2L", "U2R"]
    )
)

Multiclass KDDTest+2 Results
----------------------------
Accuracy: 0.7373580553584103

Classification Report:
              precision    recall  f1-score   support

      Normal       0.63      0.97      0.77      9711
         DoS       0.96      0.77      0.85      7458
       Probe       0.85      0.59      0.70      2421
         R2L       0.80      0.00      0.01      2887
         U2R       0.40      0.03      0.06        67

    accuracy                           0.74     22544
   macro avg       0.73      0.47      0.48     22544
weighted avg       0.79      0.74      0.69     22544

Confusion Matrix:
[[9459   67  184    0    1]
 [1686 5716   56    0    0]
 [ 823  164 1434    0    0]
 [2869    0    4   12    2]
 [  62    0    0    3    2]]


In [73]:
hierarchical_attack_rf = joblib.load(
    "../models/hierarchical_attack_rf.joblib"
)

print("Hierarchical attack classifier loaded.")

Hierarchical attack classifier loaded.


In [74]:
# Stage 1: Binary detection using frozen threshold = 0.35

y_test_binary_proba = (
    tuned_rf_binary.predict_proba(
        X_test_multi_processed
    )[:, 1]
)

y_test_binary_pred = (
    y_test_binary_proba >= 0.35
).astype(int)


# Stage 2: Attack-category classification

hierarchical_test_pred = np.empty(
    len(y_test_multi_class),
    dtype=object
)

# Start with everything classified as Normal
hierarchical_test_pred[:] = "Normal"

attack_indices = np.where(
    y_test_binary_pred == 1
)[0]

hierarchical_test_pred[attack_indices] = (
    hierarchical_attack_rf.predict(
        X_test_multi_processed[attack_indices]
    )
)

In [75]:
print("Hierarchical KDDTest+2 Results")
print("------------------------------")

print(
    classification_report(
        y_test_multi_class,
        hierarchical_test_pred,
        labels=["Normal", "DoS", "Probe", "R2L", "U2R"],
        zero_division=0
    )
)

hierarchical_test_accuracy = accuracy_score(
    y_test_multi_class,
    hierarchical_test_pred
)

hierarchical_test_macro_f1 = f1_score(
    y_test_multi_class,
    hierarchical_test_pred,
    average="macro"
)

hierarchical_test_weighted_f1 = f1_score(
    y_test_multi_class,
    hierarchical_test_pred,
    average="weighted"
)

print("Accuracy :", hierarchical_test_accuracy)
print("Macro F1 :", hierarchical_test_macro_f1)
print("Weighted F1:", hierarchical_test_weighted_f1)

Hierarchical KDDTest+2 Results
------------------------------
              precision    recall  f1-score   support

      Normal       0.71      0.97      0.82      9711
         DoS       0.96      0.84      0.90      7458
       Probe       0.82      0.80      0.81      2421
         R2L       0.98      0.10      0.19      2887
         U2R       0.69      0.16      0.27        67

    accuracy                           0.80     22544
   macro avg       0.83      0.58      0.60     22544
weighted avg       0.84      0.80      0.76     22544

Accuracy : 0.7967530163236338
Macro F1 : 0.5959125282340163
Weighted F1: 0.7616878916123208


In [76]:
hierarchical_test_cm = confusion_matrix(
    y_test_multi_class,
    hierarchical_test_pred,
    labels=["Normal", "DoS", "Probe", "R2L", "U2R"]
)

print("Hierarchical Confusion Matrix:")
print(hierarchical_test_cm)

Hierarchical Confusion Matrix:
[[9424   69  216    0    2]
 [ 980 6291  187    0    0]
 [ 318  166 1937    0    0]
 [2560    0   25  299    3]
 [  51    0    0    5   11]]


In [77]:
xgb_multiclass = joblib.load(
    "../models/xgboost_multiclass.joblib"
)

label_encoder = joblib.load(
    "../models/xgboost_multiclass_label_encoder.joblib"
)

print("XGBoost multiclass model loaded.")

XGBoost multiclass model loaded.


In [78]:
# Predict encoded class labels

y_test_xgb_encoded = xgb_multiclass.predict(
    X_test_multi_processed
)

# Convert encoded predictions back to category names

y_test_xgb_pred = label_encoder.inverse_transform(
    y_test_xgb_encoded.astype(int)
)

print("XGBoost multiclass KDDTest+2 Results")
print("-------------------------------------")

print(
    classification_report(
        y_test_multi_class,
        y_test_xgb_pred,
        labels=["Normal", "DoS", "Probe", "R2L", "U2R"],
        zero_division=0
    )
)

print(
    "Accuracy:",
    accuracy_score(
        y_test_multi_class,
        y_test_xgb_pred
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test_multi_class,
        y_test_xgb_pred,
        average="macro"
    )
)

print(
    "Weighted F1:",
    f1_score(
        y_test_multi_class,
        y_test_xgb_pred,
        average="weighted"
    )
)

XGBoost multiclass KDDTest+2 Results
-------------------------------------
              precision    recall  f1-score   support

      Normal       0.68      0.97      0.80      9711
         DoS       0.96      0.84      0.90      7458
       Probe       0.85      0.64      0.73      2421
         R2L       0.98      0.09      0.16      2887
         U2R       0.64      0.10      0.18        67

    accuracy                           0.78     22544
   macro avg       0.82      0.53      0.55     22544
weighted avg       0.83      0.78      0.74     22544

Accuracy: 0.7782114975159687
Macro F1: 0.5538015123381157
Weighted F1: 0.7413908241841173


In [79]:
xgb_test_cm = confusion_matrix(
    y_test_multi_class,
    y_test_xgb_pred,
    labels=["Normal", "DoS", "Probe", "R2L", "U2R"]
)

print("XGBoost Confusion Matrix:")
print(xgb_test_cm)

XGBoost Confusion Matrix:
[[9440   71  198    1    1]
 [1090 6296   72    0    0]
 [ 707  166 1548    0    0]
 [2627    0    4  253    3]
 [  56    0    0    4    7]]


## Testing on OOF target encoded random forest

In [3]:
import joblib

base_preprocessor = joblib.load(
    "../models/base_preprocessor_oof_target.joblib"
)

service_encoder = joblib.load(
    "../models/service_target_encoder_oof.joblib"
)

rf_oof_target = joblib.load(
    "../models/random_forest_oof_target_final.joblib"
)

print("Final supervised model and preprocessing loaded.")

Final supervised model and preprocessing loaded.


In [5]:
feature_names_original = [
    "duration",
    "protocol_type",
    "service",
    "flag",
    "src_bytes",
    "dst_bytes",
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "logged_in",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "num_outbound_cmds",
    "is_host_login",
    "is_guest_login",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "srv_diff_host_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate",
    "dst_host_srv_serror_rate",
    "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate"
]

In [7]:
attack_category_map_test = {
    # DoS
    "back": "DoS",
    "land": "DoS",
    "neptune": "DoS",
    "pod": "DoS",
    "smurf": "DoS",
    "teardrop": "DoS",
    "apache2": "DoS",
    "mailbomb": "DoS",
    "processtable": "DoS",
    "udpstorm": "DoS",

    # Probe
    "ipsweep": "Probe",
    "nmap": "Probe",
    "portsweep": "Probe",
    "satan": "Probe",
    "mscan": "Probe",
    "saint": "Probe",

    # R2L
    "ftp_write": "R2L",
    "guess_passwd": "R2L",
    "imap": "R2L",
    "multihop": "R2L",
    "phf": "R2L",
    "spy": "R2L",
    "warezclient": "R2L",
    "warezmaster": "R2L",
    "named": "R2L",
    "sendmail": "R2L",
    "snmpgetattack": "R2L",
    "snmpguess": "R2L",
    "worm": "R2L",
    "xlock": "R2L",
    "xsnoop": "R2L",
    "httptunnel": "R2L",

    # U2R
    "buffer_overflow": "U2R",
    "loadmodule": "U2R",
    "perl": "U2R",
    "rootkit": "U2R",
    "ps": "U2R",
    "sqlattack": "U2R",
    "xterm": "U2R",

    # Normal
    "normal": "Normal"
}

print("Attack category mapping loaded.")

Attack category mapping loaded.


In [8]:
test_multi_df = pd.read_csv(
    "../data/raw/KDDTest+2.txt",
    sep=",",
    header=None
)

test_multi_df.columns = feature_names_original + ["label", "difficulty"]

X_test_multi = test_multi_df[feature_names_original].copy()

y_test_multi_class = test_multi_df["label"].map(
    attack_category_map_test
)

print("Test shape:", X_test_multi.shape)
print("Target shape:", y_test_multi_class.shape)
print("\nClass distribution:")
print(y_test_multi_class.value_counts())

Test shape: (22544, 41)
Target shape: (22544,)

Class distribution:
label
Normal    9711
DoS       7458
R2L       2887
Probe     2421
U2R         67
Name: count, dtype: int64


In [9]:
X_test_base = base_preprocessor.transform(X_test_multi)

X_test_service_te = service_encoder.transform(
    X_test_multi[["service"]]
).values

X_test_oof_target = np.hstack([
    X_test_base,
    X_test_service_te
])

print("Base representation:", X_test_base.shape)
print("Service encoding:", X_test_service_te.shape)
print("Final test representation:", X_test_oof_target.shape)

Base representation: (22544, 51)
Service encoding: (22544, 5)
Final test representation: (22544, 56)


In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_pred_test_oof = rf_oof_target.predict(X_test_oof_target)

print("Accuracy:", accuracy_score(
    y_test_multi_class,
    y_pred_test_oof
))

print("Weighted Precision:", precision_score(
    y_test_multi_class,
    y_pred_test_oof,
    average="weighted",
    zero_division=0
))

print("Weighted Recall:", recall_score(
    y_test_multi_class,
    y_pred_test_oof,
    average="weighted",
    zero_division=0
))

print("Weighted F1:", f1_score(
    y_test_multi_class,
    y_pred_test_oof,
    average="weighted",
    zero_division=0
))

print("Macro F1:", f1_score(
    y_test_multi_class,
    y_pred_test_oof,
    average="macro",
    zero_division=0
))

print("\nClassification Report:")
print(classification_report(
    y_test_multi_class,
    y_pred_test_oof
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test_multi_class,
    y_pred_test_oof
))

Accuracy: 0.7636621717530163
Weighted Precision: 0.8148706818757278
Weighted Recall: 0.7636621717530163
Weighted F1: 0.7159604762026085
Macro F1: 0.5041013238407241

Classification Report:
              precision    recall  f1-score   support

         DoS       0.96      0.83      0.89      7458
      Normal       0.66      0.97      0.79      9711
       Probe       0.88      0.64      0.74      2421
         R2L       0.91      0.01      0.02      2887
         U2R       0.43      0.04      0.08        67

    accuracy                           0.76     22544
   macro avg       0.77      0.50      0.50     22544
weighted avg       0.81      0.76      0.72     22544


Confusion Matrix:
[[6171 1264   23    0    0]
 [  67 9458  185    0    1]
 [ 164  702 1555    0    0]
 [   0 2850    5   29    3]
 [   0   61    0    3    3]]
